In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version PyTorch is using: {torch.version.cuda}")
print(f"Is the GPU recognized? {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu130
CUDA version PyTorch is using: 13.0
Is the GPU recognized? True


In [3]:
pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 9.5 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

## Configuration

In [3]:
ZIP_PATH = Path("train_processed_final.zip")     
LABELS_CSV = Path("trainLabels.csv")             
EXTRACT_DIR = Path("train_processed_data")       
MODEL_SAVE_PATH = Path("best_convnext_tiny.pth")

SEED = 42
NUM_CLASSES = 5
BATCH_SIZE = 16
NUM_EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_SIZE = 0.2
NUM_WORKERS = 4
IMG_SIZE = 224

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [6]:
if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Dataset extracted.")
else:
    print("Dataset already extracted.")

Dataset extracted.


In [7]:
labels_df = pd.read_csv(LABELS_CSV)
print(labels_df.head())
print(labels_df.columns.tolist())

      image  level
0   10_left      0
1  10_right      0
2   13_left      0
3  13_right      0
4   15_left      1
['image', 'level']


In [8]:
image_files = list(EXTRACT_DIR.glob("*.jpg")) + list(EXTRACT_DIR.glob("*.jpeg")) + list(EXTRACT_DIR.glob("*.png"))
print("Total extracted image files:", len(image_files))
image_map = {img.stem: img for img in image_files}
print("Mapped image stems:", len(image_map))

Total extracted image files: 35126
Mapped image stems: 35126


In [9]:
labels_df["image_path"] = labels_df["image"].map(image_map)
labels_df = labels_df.dropna(subset=["image_path"]).copy()

labels_df["image_path"] = labels_df["image_path"].astype(str)
labels_df["level"] = labels_df["level"].astype(int)

print("Matched samples:", len(labels_df))
print(labels_df["level"].value_counts().sort_index())
labels_df.head()

Matched samples: 35126
level
0    25810
1     2443
2     5292
3      873
4      708
Name: count, dtype: int64


,image,level,image_path
0,10_left,0,train_processed_data\10_left.jpg
1,10_right,0,train_processed_data\10_right.jpg
2,13_left,0,train_processed_data\13_left.jpg
3,13_right,0,train_processed_data\13_right.jpg
4,15_left,1,train_processed_data\15_left.jpg


In [10]:
train_df, val_df = train_test_split(
    labels_df,
    test_size = VAL_SIZE,
    stratify = labels_df["level"],
    random_state = SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

Train size: 28100
Val size: 7026


In [14]:
train_df.to_csv("train_dataset",index = False)
val_df.to_csv("val_dataset",index = False)